In [ ]:
# Setup Imports and Load Data

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.signal import savgol_filter
from scipy.interpolate import PchipInterpolator

OUT = Path('output/visualizations_testing')
IN = Path('source_data')
FIELDS = [87, 96, 99, 103, 74, 87, 96.2, 151, 176]

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 8,
    'axes.linewidth': 1.0,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'axes.xmargin' : 0.05,
    'axes.ymargin' : 0.05
})

# Loading data and returns T (array of temperatures, x-axis), nu (array of fillings), and R (2D array of resistivity)
def load_field(E):
    df = pd.read_csv(IN / f'Rxx_matrix_E-{E}mV_nm.csv')
    T = df.iloc[:, 0].astype(float).to_numpy()
    nu = np.array([float(c) for c in df.columns[1:]])
    R = df.iloc[:, 1:].astype(float).to_numpy()
    return T, nu, R

def fmt4(x):
    if x == 0:
        return "0.000"

    digits_before = 1 if abs(x) < 1 else int(math.log10(abs(x))) + 1
    decimals = max(0, 4 - digits_before)
    return f"{x:.{decimals}f}"

In [ ]:
# Smoothing function

def smooth_rho(T, rho, window_frac=0.08, polyorder=2):
    """
    Lightly smooth rho(T) when T is unevenly spaced.

    Same signature as before:
        smooth_rho(T, rho, window_frac=0.08, polyorder=2)

    Difference:
        window_frac is now interpreted as a fraction of the T-range,
        not a fraction of the number of indices.
    """

    T = np.asarray(T, dtype=float)
    rho = np.asarray(rho, dtype=float)

    rho_out = np.full_like(rho, np.nan, dtype=float)

    mask = np.isfinite(T) & np.isfinite(rho)
    if np.sum(mask) < polyorder + 3:
        return rho.copy()

    T_valid = T[mask]
    rho_valid = rho[mask]

    order = np.argsort(T_valid)
    T_sorted = T_valid[order]
    rho_sorted = rho_valid[order]

    # Combine duplicate T values using median rho
    unique_T = np.unique(T_sorted)
    if len(unique_T) < len(T_sorted):
        rho_unique = np.array([
            np.median(rho_sorted[T_sorted == t]) for t in unique_T
        ])
        T_sorted = unique_T
        rho_sorted = rho_unique

    if len(T_sorted) < polyorder + 3:
        rho_out[mask] = rho_valid
        return rho_out

    dT = np.diff(T_sorted)
    dT = dT[dT > 0]

    if len(dT) == 0:
        rho_out[mask] = rho_valid
        return rho_out

    # Uniform grid spacing based on actual temperature spacing
    grid_step = np.median(dT)

    T_grid = np.arange(
        T_sorted.min(),
        T_sorted.max() + grid_step,
        grid_step
    )

    # Interpolate uneven data onto uniform T grid
    interp_raw = PchipInterpolator(T_sorted, rho_sorted)
    rho_grid = interp_raw(T_grid)

    # Convert fractional T-window into SavGol index-window
    T_range = T_sorted.max() - T_sorted.min()
    window_T = window_frac * T_range

    window_length = int(round(window_T / grid_step))

    # SavGol requirements
    window_length = max(window_length, polyorder + 3)
    if window_length % 2 == 0:
        window_length += 1

    if window_length >= len(T_grid):
        window_length = len(T_grid) - 1
        if window_length % 2 == 0:
            window_length -= 1

    if window_length <= polyorder:
        rho_out[mask] = rho_valid
        return rho_out

    # Smooth on uniform grid
    rho_grid_smooth = savgol_filter(
        rho_grid,
        window_length=window_length,
        polyorder=polyorder,
        mode="interp"
    )

    # Interpolate smoothed curve back to original uneven T points
    interp_smooth = PchipInterpolator(T_grid, rho_grid_smooth)
    rho_smooth_valid = interp_smooth(T_valid)

    rho_out[mask] = rho_smooth_valid

    return rho_out


In [ ]:
# Showing 1st and 2nd deriveratives for smoothed data

import os

OUT = Path('output/visualizations_testing/1st_and_2nd_derivs')
if not os.path.exists(OUT):
    os.makedirs(OUT)

def plot_linecut(params: dict, T: np.ndarray, rho: np.ndarray) -> None:

    keys = params.keys()
    param_string = ""

    for key in keys:
        value = params[key]
        value = fmt4(value)
        string = str(key) + " = " + str(value) + "  "
        param_string += string

    scale = 8
    fig, axes = plt.subplots(2, 3, figsize=(2.2 * scale, scale), dpi=180)
    fig.suptitle(param_string)

    for ax in axes.flatten():
        ax.set_ylabel("Resistivity (Ω·cm)") 
        ax.set_xlabel("Temperature (K)")

    (ax1, ax2, ax3), (ax4, ax5, ax6) = axes

    ax1.set_title("p(T)")
    ax2.set_title("dp/dT")
    ax3.set_title("d2p/dT2")
    ax4.set_title("Smoothed p(T)")
    ax5.set_title("dp/dT of smoothed p")
    ax6.set_title("d2p/dT2 of smoothed p")

    dpdT = np.gradient(rho, T)
    d2pdT2 = np.gradient(dpdT, T)
    rho_smoothed = smooth_rho(T, rho)
    dpdT_smoothed = np.gradient(rho_smoothed, T)
    d2pdT2_smoothed = np.gradient(dpdT_smoothed, T)

    ax1.plot(T, rho, marker='o', markerfacecolor='none', markeredgecolor='navy')
    ax1.fill_between(T, rho, alpha = 0.5)

    ax2.plot(T, dpdT, marker='o', markerfacecolor='none', markeredgecolor='navy')
    ax2.fill_between(T, dpdT, alpha = 0.5)

    ax3.plot(T, d2pdT2, marker='o', markerfacecolor='none', markeredgecolor='navy')
    ax3.fill_between(T, d2pdT2, alpha = 0.5)

    ax4.plot(T, rho_smoothed, marker='o', markerfacecolor='none', markeredgecolor='navy')
    ax4.fill_between(T, rho_smoothed, alpha = 0.5)

    ax5.plot(T, dpdT_smoothed, marker='o', markerfacecolor='none', markeredgecolor='navy')
    ax5.fill_between(T, dpdT_smoothed, alpha = 0.5)

    ax6.plot(T, d2pdT2_smoothed, marker='o', markerfacecolor='none', markeredgecolor='navy')
    ax6.fill_between(T, d2pdT2_smoothed, alpha = 0.5)

    # Saving plots to path
    path = OUT / Path(param_string + ".png")
    fig.savefig(path, dpi=250, bbox_inches='tight')


# Plots evenly spread linecuts
def plot_all_linecuts(E: int, numLines: int) -> None:
    T, nu, R = load_field(E)

    row, col = R.shape
    currCol = 0
    spacing = (col - 1) / (numLines - 1)

    for _ in range(numLines):
        currColInt = int(currCol + 0.5)
        linecut_rho = R[:, currColInt]

        # Plotting the intervals
        plot_linecut({"E" : E, "Filling" : nu[currColInt]}, T, linecut_rho)

        currCol += spacing

plot_all_linecuts(103, 5)

In [ ]:
# HAMPEL showcase

from hampel import hampel
import os

OUT = Path('output/visualizations_testing/hampel')

if not os.path.exists(OUT):
    os.makedirs(OUT)

def plot_linecut(params: dict, T: np.ndarray, rho: np.ndarray) -> None:

    keys = params.keys()
    param_string = ""

    for key in keys:
        value = params[key]
        value = fmt4(value)
        string = str(key) + " = " + str(value) + "  "
        param_string += string

    scale = 8
    fig, axes = plt.subplots(1, 3, figsize=(2.2 * scale, scale), dpi=180)
    fig.suptitle(param_string)



    ax1, ax2, ax3 = axes

    ax1.set_title("Raw p(T)")
    ax2.set_title("Hampel Filter")
    ax3.set_title("Smooth after Hampel")

    rho_hampel = hampel(rho).filtered_data
    rho_smoothed = smooth_rho(T, rho_hampel)

    ax1.plot(T, rho, marker='o', markerfacecolor='none', markeredgecolor='navy')
    ax2.plot(T, rho_hampel, marker='o', markerfacecolor='none', markeredgecolor='navy')
    ax3.plot(T, rho_smoothed, marker='o', markerfacecolor='none', markeredgecolor='navy')

    for ax in axes.flatten():
        ax.set_ylabel("Resistivity (Ω·cm)") 
        ax.set_xlabel("Temperature (K)")
        ax.set_xlim(left=0)
        ax.set_ylim(bottom=0)


    plt.show()
    # Saving plots to path
    path = OUT / Path(param_string + ".png")
    fig.savefig(path, dpi=250, bbox_inches='tight')


# Plots evenly spread linecuts
def plot_all_linecuts(E: int, numLines: int) -> None:
    T, nu, R = load_field(E)

    row, col = R.shape
    currCol = 0
    spacing = (col - 1) / (numLines - 1)

    for _ in range(numLines):
        currColInt = int(currCol + 0.5)
        linecut_rho = R[:, currColInt]

        # Plotting the intervals
        plot_linecut({"E" : E, "Filling" : nu[currColInt]}, T, linecut_rho)

        currCol += spacing


plot_all_linecuts(103, 20)

In [ ]:
# HAMPEL Signal screening

def plot_linecut(params: dict, T: np.ndarray, rho: np.ndarray) -> None:

    keys = params.keys()
    param_string = "Hampel Smoothing  "

    for key in keys:
        value = params[key]
        value = fmt4(value)
        string = str(key) + " = " + str(value) + "  "
        param_string += string

    scale = 8
    fig, axes = plt.subplots(1, 3, figsize=(2.2 * scale, scale), dpi=180)
    fig.suptitle(param_string)

    ax1, ax2, ax3 = axes

    ax1.set_title("Raw p(T)")
    ax2.set_title("Hampel Filter")
    ax3.set_title("Smooth after Hampel")

    rho_hampel = hampel(rho).filtered_data
    rho_smoothed = smooth_rho(T, rho_hampel)

    ax1.plot(T, rho, marker='o', markerfacecolor='none', markeredgecolor='navy')
    ax2.plot(T, rho_hampel, marker='o', markerfacecolor='none', markeredgecolor='navy')
    ax3.plot(T, rho_smoothed, marker='o', markerfacecolor='none', markeredgecolor='navy')

    for ax in axes.flatten():
        ax.set_ylabel("Resistivity (Ω·cm)") 
        ax.set_xlabel("Temperature (K)")
        ax.set_xlim(left=0)
        ax.set_ylim(bottom=0)

    # Saving plots to path
    path = OUT / Path(param_string + ".png")
    fig.savefig(path, dpi=250, bbox_inches='tight')


# Plots evenly spread linecuts
def plot_all_linecuts(E: int, numLines: int) -> None:
    T, nu, R = load_field(E)

    row, col = R.shape
    currCol = 0
    spacing = (col - 1) / (numLines - 1)

    for _ in range(numLines):
        currColInt = int(currCol + 0.5)
        linecut_rho = R[:, currColInt]

        # Plotting the intervals
        plot_linecut({"E" : E, "Filling" : nu[currColInt]}, T, linecut_rho)

        currCol += spacing


plot_all_linecuts(103, 20)